# Saúde+ Analytics — SQL Validation Notebook
**Case Técnico | Time de Dados**

Este notebook contém todos os SQLs que correspondem a cada visualização do dashboard Streamlit.
Execute com DuckDB (zero config) ou adapte para Snowflake / BigQuery / Databricks SQL.

---
**Estrutura:**
- Setup & Conexão
- Q1 — Prescrições Diárias & Sazonalidade
- Q2 — Pacientes Atendidos
- Q3 — Especialidades que Mais Prescrevem
- Q4 — Open Rate
- Q5 — Conversão por Canal
- Q6 — Insights Adicionais (Medicamentos, Tempo de Compra, Retenção)
- Q7 — Recomendações Operacionais (SQLs de suporte)

## 0. Setup — DuckDB + Carregamento dos CSVs

In [1]:
# pip install duckdb pandas
import duckdb
import pandas as pd

con = duckdb.connect()

# ── Carrega os 3 arquivos CSV como views ──────────────────────────────────────
# Ajuste os caminhos se necessário
DATA_DIR = './data'  # pasta onde estão os CSVs

con.execute(f"""
CREATE OR REPLACE VIEW prescricao AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/prescricaomedicamento.csv', header=true);
""")

con.execute(f"""
CREATE OR REPLACE VIEW medicamentos AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/medicamentos.csv', header=true);
""")

con.execute(f"""
CREATE OR REPLACE VIEW medicos AS
    SELECT * FROM read_csv_auto('{DATA_DIR}/medicos.csv', header=true);
""")

print('Views criadas com sucesso.')
print('prescricao:', con.execute('SELECT COUNT(*) FROM prescricao').fetchone()[0], 'linhas')
print('medicamentos:', con.execute('SELECT COUNT(*) FROM medicamentos').fetchone()[0], 'linhas')
print('medicos:', con.execute('SELECT COUNT(*) FROM medicos').fetchone()[0], 'linhas')

Views criadas com sucesso.
prescricao: 73929 linhas
medicamentos: 22 linhas
medicos: 45025 linhas


In [2]:
# Helper para rodar SQL e retornar DataFrame
def sql(query, preview=10):
    df = con.execute(query).df()
    print(f'{len(df)} linhas  |  {list(df.columns)}')
    return df.head(preview)

---
## 1. Schema Exploration — Estrutura das Tabelas

In [3]:
# Estrutura da tabela principal
sql("""
SELECT column_name, column_type
FROM information_schema.columns
WHERE table_name = 'prescricao'
ORDER BY ordinal_position
""")

BinderException: Binder Error: Referenced column "column_type" not found in FROM clause!
Candidate bindings: "column_name", "column_default", "data_type", "collation_name", "scope_name"

LINE 2: SELECT column_name, column_type
                            ^

In [4]:
# Amostra da tabela principal
sql("""
SELECT
    idprescricao,
    idmedicamento,
    dataprescricao::TIMESTAMP AS dataprescricao,
    idmedico,
    idpaciente,
    visualizadapaciente,
    itemvendido,
    canalvenda,
    sexopaciente,
    estadopaciente
FROM prescricao
LIMIT 5
""")

5 linhas  |  ['idprescricao', 'idmedicamento', 'dataprescricao', 'idmedico', 'idpaciente', 'visualizadapaciente', 'itemvendido', 'canalvenda', 'sexopaciente', 'estadopaciente']


,idprescricao,idmedicamento,dataprescricao,idmedico,idpaciente,visualizadapaciente,itemvendido,canalvenda,sexopaciente,estadopaciente
0,65286505,3783,2025-01-28 13:00:11,115888,16277,True,0,não convertido,Masculino,SP
1,65053805,14714,2025-01-24 13:02:29,41433,34330,False,0,não convertido,Feminino,SP
2,69631101,3783,2025-03-26 15:00:26,77184,10831,True,1,farmacia fisica,Feminino,SP
3,67746000,11191,2025-02-28 23:31:13,107252,54414,False,0,não convertido,Masculino,NaN
4,68167102,3783,2025-03-08 13:35:08,30764,63594,True,0,não convertido,Masculino,SP


In [5]:
# Distribuição dos valores de canalvenda
sql("""
SELECT
    canalvenda,
    COUNT(*) AS linhas,
    COUNT(DISTINCT idprescricao) AS prescricoes
FROM prescricao
GROUP BY 1
ORDER BY 2 DESC
""")

3 linhas  |  ['canalvenda', 'linhas', 'prescricoes']


,canalvenda,linhas,prescricoes
0,não convertido,68745,66306
1,farmacia fisica,4411,3888
2,marketplace,773,728


---
## Q1 — Prescrições Diárias & Sazonalidade

In [6]:
# ── Q1.1 · Volume diário de prescrições ──────────────────────────────────────
# Nível: prescrição única por dia (deduplica idprescricao)
sql("""
SELECT
    CAST(dataprescricao AS DATE) AS data,
    COUNT(DISTINCT idprescricao) AS prescricoes
FROM prescricao
GROUP BY 1
ORDER BY 1
""", preview=15)

108 linhas  |  ['data', 'prescricoes']


,data,prescricoes
0,2025-01-01,171
1,2025-01-02,534
2,2025-01-03,557
3,2025-01-04,348
4,2025-01-05,251
5,2025-01-06,621
6,2025-01-07,712
7,2025-01-08,755
8,2025-01-09,731
9,2025-01-10,655


In [7]:
# ── Q1.2 · Volume semanal (usado no gráfico de barras do dashboard) ───────────
sql("""
SELECT
    DATE_TRUNC('week', dataprescricao::TIMESTAMP) AS semana_inicio,
    COUNT(DISTINCT idprescricao) AS prescricoes
FROM prescricao
GROUP BY 1
ORDER BY 1
""")

16 linhas  |  ['semana_inicio', 'prescricoes']


,semana_inicio,prescricoes
0,2024-12-30,1861
1,2025-01-06,4052
2,2025-01-13,4059
3,2025-01-20,3759
4,2025-01-27,3944
5,2025-02-03,3884
6,2025-02-10,4237
7,2025-02-17,4850
8,2025-02-24,4709
9,2025-03-03,3796


In [8]:
# ── Q1.3 · Média diária, mínimo, máximo ──────────────────────────────────────
sql("""
WITH diario AS (
    SELECT
        CAST(dataprescricao AS DATE) AS data,
        COUNT(DISTINCT idprescricao) AS prescricoes
    FROM prescricao
    GROUP BY 1
)
SELECT
    MIN(data)          AS data_inicio,
    MAX(data)          AS data_fim,
    COUNT(*)           AS dias_com_movimento,
    SUM(prescricoes)   AS total_prescricoes,
    ROUND(AVG(prescricoes), 1) AS media_diaria,
    MIN(prescricoes)   AS min_diario,
    MAX(prescricoes)   AS max_diario
FROM diario
""")

1 linhas  |  ['data_inicio', 'data_fim', 'dias_com_movimento', 'total_prescricoes', 'media_diaria', 'min_diario', 'max_diario']


,data_inicio,data_fim,dias_com_movimento,total_prescricoes,media_diaria,min_diario,max_diario
0,2025-01-01,2025-04-18,108,70909.0,656.6,35,1271


In [9]:
# ── Q1.4 · Sazonalidade por dia da semana ────────────────────────────────────
sql("""
SELECT
    DAYNAME(dataprescricao::TIMESTAMP) AS dia_semana,
    DAYOFWEEK(dataprescricao::TIMESTAMP) AS num_dia,
    COUNT(DISTINCT idprescricao) AS prescricoes,
    ROUND(COUNT(DISTINCT idprescricao) * 100.0 /
          SUM(COUNT(DISTINCT idprescricao)) OVER (), 1) AS pct_total
FROM prescricao
GROUP BY 1, 2
ORDER BY 2
""")

7 linhas  |  ['dia_semana', 'num_dia', 'prescricoes', 'pct_total']


,dia_semana,num_dia,prescricoes,pct_total
0,Sunday,0,4194,5.9
1,Monday,1,12147,17.1
2,Tuesday,2,12477,17.6
3,Wednesday,3,12638,17.8
4,Thursday,4,13071,18.4
5,Friday,5,10720,15.1
6,Saturday,6,5662,8.0


In [10]:
# ── Q1.5 · Sazonalidade por hora do dia ──────────────────────────────────────
sql("""
SELECT
    HOUR(dataprescricao::TIMESTAMP) AS hora,
    COUNT(DISTINCT idprescricao)    AS prescricoes
FROM prescricao
GROUP BY 1
ORDER BY 1
""")

24 linhas  |  ['hora', 'prescricoes']


,hora,prescricoes
0,0,1895
1,1,1331
2,2,1126
3,3,750
4,4,501
5,5,358
6,6,265
7,7,225
8,8,280
9,9,577


In [11]:
# ── Q1.6 · Crescimento mês a mês ─────────────────────────────────────────────
sql("""
WITH mensal AS (
    SELECT
        DATE_TRUNC('month', dataprescricao::TIMESTAMP) AS mes,
        COUNT(DISTINCT idprescricao) AS prescricoes
    FROM prescricao
    GROUP BY 1
)
SELECT
    mes,
    prescricoes,
    LAG(prescricoes) OVER (ORDER BY mes) AS mes_anterior,
    ROUND(
        (prescricoes - LAG(prescricoes) OVER (ORDER BY mes)) * 100.0
        / NULLIF(LAG(prescricoes) OVER (ORDER BY mes), 0),
    1) AS crescimento_pct
FROM mensal
ORDER BY 1
""")

4 linhas  |  ['mes', 'prescricoes', 'mes_anterior', 'crescimento_pct']


,mes,prescricoes,mes_anterior,crescimento_pct
0,2025-01-01,17176,<NA>,NaN
1,2025-02-01,17577,17176,2.3
2,2025-03-01,20907,17577,18.9
3,2025-04-01,15247,20907,-27.1


---
## Q2 — Pacientes Atendidos

In [12]:
# ── Q2.1 · Total de pacientes únicos ─────────────────────────────────────────
sql("""
SELECT
    COUNT(DISTINCT idpaciente) AS pacientes_unicos,
    COUNT(DISTINCT idprescricao) AS prescricoes_unicas,
    ROUND(COUNT(DISTINCT idprescricao) * 1.0 / COUNT(DISTINCT idpaciente), 2)
        AS prescricoes_por_paciente
FROM prescricao
""")

1 linhas  |  ['pacientes_unicos', 'prescricoes_unicas', 'prescricoes_por_paciente']


,pacientes_unicos,prescricoes_unicas,prescricoes_por_paciente
0,68767,70907,1.03


In [13]:
# ── Q2.2 · Distribuição por sexo (1 linha por paciente — usa primeira ocorrência)
sql("""
WITH paciente_sexo AS (
    SELECT
        idpaciente,
        -- normaliza variações do campo sexo
        CASE
            WHEN UPPER(sexopaciente) IN ('FEMININO','F') THEN 'Feminino'
            WHEN UPPER(sexopaciente) IN ('MASCULINO','M') THEN 'Masculino'
            ELSE 'Não Informado'
        END AS sexo
    FROM (
        SELECT idpaciente, sexopaciente,
               ROW_NUMBER() OVER (PARTITION BY idpaciente ORDER BY dataprescricao) AS rn
        FROM prescricao
    ) t
    WHERE rn = 1
)
SELECT
    sexo,
    COUNT(*)  AS pacientes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM paciente_sexo
GROUP BY 1
ORDER BY 2 DESC
""")

3 linhas  |  ['sexo', 'pacientes', 'pct']


,sexo,pacientes,pct
0,Feminino,40273,58.6
1,Masculino,25168,36.6
2,Não Informado,3326,4.8


In [14]:
# ── Q2.3 · Top 10 estados por pacientes únicos ───────────────────────────────
sql("""
WITH paciente_estado AS (
    SELECT DISTINCT idpaciente, estadopaciente
    FROM (
        SELECT idpaciente, estadopaciente,
               ROW_NUMBER() OVER (PARTITION BY idpaciente ORDER BY dataprescricao) AS rn
        FROM prescricao
    ) t
    WHERE rn = 1
)
SELECT
    estadopaciente AS estado,
    COUNT(*)        AS pacientes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM paciente_estado
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10
""")

10 linhas  |  ['estado', 'pacientes', 'pct']


,estado,pacientes,pct
0,SP,33550,48.8
1,NaN,7575,11.0
2,ES,3949,5.7
3,PR,3842,5.6
4,MG,3717,5.4
5,RS,3401,4.9
6,RJ,3185,4.6
7,SC,3070,4.5
8,RN,1326,1.9
9,DF,975,1.4


In [15]:
# ── Q2.4 · Pacientes com mais de 1 prescrição (retenção) ─────────────────────
sql("""
WITH contagem AS (
    SELECT
        idpaciente,
        COUNT(DISTINCT idprescricao) AS num_prescricoes
    FROM prescricao
    GROUP BY 1
)
SELECT
    num_prescricoes,
    COUNT(*)  AS pacientes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM contagem
GROUP BY 1
ORDER BY 1
LIMIT 10
""")

4 linhas  |  ['num_prescricoes', 'pacientes', 'pct']


,num_prescricoes,pacientes,pct
0,1,66749,97.1
1,2,1908,2.8
2,3,98,0.1
3,4,12,0.0


---
## Q3 — Especialidades que Mais Prescrevem

In [16]:
# ── Q3.1 · Ranking de especialidades por prescrições únicas ──────────────────
sql("""
SELECT
    m.especialidade,
    COUNT(DISTINCT p.idprescricao) AS prescricoes,
    COUNT(DISTINCT p.idmedico)     AS medicos_ativos,
    COUNT(DISTINCT p.idpaciente)   AS pacientes_atendidos,
    ROUND(COUNT(DISTINCT p.idprescricao) * 100.0
          / SUM(COUNT(DISTINCT p.idprescricao)) OVER (), 1) AS pct_total
FROM prescricao p
LEFT JOIN medicos m ON p.idmedico = m.idmedico
GROUP BY 1
ORDER BY 2 DESC
LIMIT 15
""")

15 linhas  |  ['especialidade', 'prescricoes', 'medicos_ativos', 'pacientes_atendidos', 'pct_total']


,especialidade,prescricoes,medicos_ativos,pacientes_atendidos,pct_total
0,CLINICA MEDICA,15293,3083,15035,21.6
1,SEM ESPECIALIDADE,12959,2457,12668,18.3
2,PEDIATRIA,8702,1968,8471,12.3
3,CIRURGIA GERAL,5380,958,5307,7.6
4,ORTOPEDIA E TRAUMATOLOGIA,4473,947,4420,6.3
5,MEDICINA DE FAMILIA E COMUNIDADE,2804,530,2734,4.0
6,GINECOLOGIA E OBSTETRICIA,2687,999,2634,3.8
7,PSIQUIATRIA,2633,507,2446,3.7
8,ENDOCRINOLOGIA E METABOLOGIA,2088,292,2034,2.9
9,CARDIOLOGIA,1945,510,1894,2.7


In [17]:
# ── Q3.2 · Open Rate e Conversão por especialidade ───────────────────────────
# Nível de cálculo: por prescrição única
sql("""
WITH presc_nivel AS (
    -- deduplica para nível de prescrição
    SELECT
        p.idprescricao,
        m.especialidade,
        MAX(p.visualizadapaciente::INT)  AS visualizada,
        MAX(p.itemvendido)               AS itens_vendidos
    FROM prescricao p
    LEFT JOIN medicos m ON p.idmedico = m.idmedico
    GROUP BY 1, 2
)
SELECT
    especialidade,
    COUNT(*)                                         AS total_prescricoes,
    SUM(visualizada)                                 AS visualizadas,
    SUM(CASE WHEN itens_vendidos > 0 THEN 1 ELSE 0 END) AS vendidas,
    ROUND(SUM(visualizada) * 100.0 / COUNT(*), 1)   AS open_rate_pct,
    ROUND(
        SUM(CASE WHEN itens_vendidos > 0 THEN 1 ELSE 0 END) * 100.0
        / NULLIF(SUM(visualizada), 0),
    1) AS conversao_pct
FROM presc_nivel
GROUP BY 1
HAVING COUNT(*) >= 50
ORDER BY 2 DESC
LIMIT 12
""")

12 linhas  |  ['especialidade', 'total_prescricoes', 'visualizadas', 'vendidas', 'open_rate_pct', 'conversao_pct']


,especialidade,total_prescricoes,visualizadas,vendidas,open_rate_pct,conversao_pct
0,CLINICA MEDICA,15293,6605.0,871.0,43.2,13.2
1,SEM ESPECIALIDADE,12959,7534.0,894.0,58.1,11.9
2,PEDIATRIA,8702,4142.0,382.0,47.6,9.2
3,CIRURGIA GERAL,5380,3862.0,392.0,71.8,10.2
4,ORTOPEDIA E TRAUMATOLOGIA,4473,1849.0,337.0,41.3,18.2
5,MEDICINA DE FAMILIA E COMUNIDADE,2804,1685.0,215.0,60.1,12.8
6,GINECOLOGIA E OBSTETRICIA,2687,1219.0,149.0,45.4,12.2
7,PSIQUIATRIA,2633,1502.0,431.0,57.0,28.7
8,ENDOCRINOLOGIA E METABOLOGIA,2088,943.0,78.0,45.2,8.3
9,CARDIOLOGIA,1945,850.0,79.0,43.7,9.3


In [18]:
# ── Q3.3 · Perfil dos médicos (gênero e faixa etária) ────────────────────────
sql("""
SELECT
    genero,
    COUNT(*)  AS medicos,
    ROUND(AVG(idade), 1) AS idade_media,
    MIN(idade) AS idade_min,
    MAX(idade) AS idade_max
FROM medicos
WHERE idade IS NOT NULL
GROUP BY 1
ORDER BY 2 DESC
""")

2 linhas  |  ['genero', 'medicos', 'idade_media', 'idade_min', 'idade_max']


,genero,medicos,idade_media,idade_min,idade_max
0,M,20,53.1,40.0,69.0
1,F,9,44.7,38.0,55.0


In [19]:
# ── Q3.4 · Top estados dos médicos ───────────────────────────────────────────
sql("""
SELECT
    estado,
    COUNT(DISTINCT idmedico)  AS medicos,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM medicos
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10
""")

10 linhas  |  ['estado', 'medicos', 'pct']


,estado,medicos,pct
0,NaN,23983,53.3
1,SP,9531,21.2
2,ES,1778,3.9
3,RS,1583,3.5
4,MG,1525,3.4
5,PR,1215,2.7
6,DF,774,1.7
7,RJ,642,1.4
8,SC,526,1.2
9,BA,524,1.2


---
## Q4 — Open Rate

In [20]:
# ── Q4.1 · Open Rate geral ───────────────────────────────────────────────────
# Fórmula: prescrições visualizadas / total de prescrições
# Nível: idprescricao (MAX da flag por prescrição)
sql("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(visualizadapaciente::INT) AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    COUNT(*)                                AS total_prescricoes,
    SUM(visualizada)                        AS visualizadas,
    COUNT(*) - SUM(visualizada)             AS nao_visualizadas,
    ROUND(SUM(visualizada) * 100.0 / COUNT(*), 2) AS open_rate_pct
FROM presc_nivel
""")

1 linhas  |  ['total_prescricoes', 'visualizadas', 'nao_visualizadas', 'open_rate_pct']


,total_prescricoes,visualizadas,nao_visualizadas,open_rate_pct
0,70907,35772.0,35135.0,50.45


In [21]:
# ── Q4.2 · Open Rate semanal (tendência) ─────────────────────────────────────
sql("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        DATE_TRUNC('week', MIN(dataprescricao::TIMESTAMP)) AS semana,
        MAX(visualizadapaciente::INT) AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    semana,
    COUNT(*)                        AS prescricoes,
    SUM(visualizada)                AS visualizadas,
    ROUND(SUM(visualizada) * 100.0 / COUNT(*), 1) AS open_rate_pct
FROM presc_nivel
GROUP BY 1
ORDER BY 1
""")

16 linhas  |  ['semana', 'prescricoes', 'visualizadas', 'open_rate_pct']


,semana,prescricoes,visualizadas,open_rate_pct
0,2024-12-30,1861,921.0,49.5
1,2025-01-06,4052,2084.0,51.4
2,2025-01-13,4059,2015.0,49.6
3,2025-01-20,3759,1921.0,51.1
4,2025-01-27,3944,2060.0,52.2
5,2025-02-03,3884,1967.0,50.6
6,2025-02-10,4237,2187.0,51.6
7,2025-02-17,4850,2452.0,50.6
8,2025-02-24,4709,2364.0,50.2
9,2025-03-03,3796,1944.0,51.2


In [22]:
# ── Q4.3 · Open Rate por dia da semana de emissão ────────────────────────────
sql("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        DAYNAME(MIN(dataprescricao::TIMESTAMP))    AS dia_semana,
        DAYOFWEEK(MIN(dataprescricao::TIMESTAMP))  AS num_dia,
        MAX(visualizadapaciente::INT) AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    dia_semana,
    num_dia,
    COUNT(*)   AS prescricoes,
    ROUND(AVG(visualizada) * 100, 1) AS open_rate_pct
FROM presc_nivel
GROUP BY 1, 2
ORDER BY 2
""")

7 linhas  |  ['dia_semana', 'num_dia', 'prescricoes', 'open_rate_pct']


,dia_semana,num_dia,prescricoes,open_rate_pct
0,Sunday,0,4194,48.6
1,Monday,1,12147,51.8
2,Tuesday,2,12477,50.6
3,Wednesday,3,12638,50.5
4,Thursday,4,13071,50.1
5,Friday,5,10719,49.6
6,Saturday,6,5661,50.8


In [23]:
# ── Q4.4 · Open Rate por hora de emissão ─────────────────────────────────────
sql("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        HOUR(MIN(dataprescricao::TIMESTAMP)) AS hora,
        MAX(visualizadapaciente::INT) AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    hora,
    COUNT(*)  AS prescricoes,
    ROUND(AVG(visualizada) * 100, 1) AS open_rate_pct
FROM presc_nivel
GROUP BY 1
ORDER BY 1
""")

24 linhas  |  ['hora', 'prescricoes', 'open_rate_pct']


,hora,prescricoes,open_rate_pct
0,0,1893,49.2
1,1,1331,45.8
2,2,1126,42.8
3,3,749,41.5
4,4,501,41.7
5,5,357,37.0
6,6,265,37.4
7,7,225,39.1
8,8,280,51.1
9,9,577,61.5


In [24]:
# ── Q4.5 · Open Rate por estado do paciente ──────────────────────────────────
sql("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(estadopaciente)           AS estado,
        MAX(visualizadapaciente::INT) AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    estado,
    COUNT(*)  AS prescricoes,
    ROUND(AVG(visualizada) * 100, 1) AS open_rate_pct,
    ROUND((1 - AVG(visualizada)) * 100, 1) AS nao_abertura_pct
FROM presc_nivel
GROUP BY 1
HAVING COUNT(*) >= 30
ORDER BY nao_abertura_pct DESC
LIMIT 15
""")

15 linhas  |  ['estado', 'prescricoes', 'open_rate_pct', 'nao_abertura_pct']


,estado,prescricoes,open_rate_pct,nao_abertura_pct
0,TO,233,30.9,69.1
1,AM,538,32.3,67.7
2,SE,435,35.2,64.8
3,MG,3806,35.7,64.3
4,ES,4054,37.5,62.5
5,Pr,40,37.5,62.5
6,PR,3955,39.2,60.8
7,RN,1359,39.3,60.7
8,RO,97,46.4,53.6
9,BA,968,47.6,52.4


---
## Q5 — Conversão por Canal

In [25]:
# ── Q5.1 · Funil completo: emitidas → visualizadas → vendidas ─────────────────
sql("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(visualizadapaciente::INT)         AS visualizada,
        MAX(CASE WHEN itemvendido > 0 THEN 1 ELSE 0 END) AS vendida
    FROM prescricao
    GROUP BY 1
)
SELECT
    COUNT(*)          AS emitidas,
    SUM(visualizada)  AS visualizadas,
    SUM(vendida)      AS vendidas,
    -- Rates
    ROUND(SUM(visualizada) * 100.0 / COUNT(*), 1)            AS open_rate_pct,
    ROUND(SUM(vendida) * 100.0 / NULLIF(SUM(visualizada),0), 1) AS conversao_pct,
    ROUND(SUM(vendida) * 100.0 / COUNT(*), 1)                AS conversao_total_pct
FROM presc_nivel
""")

1 linhas  |  ['emitidas', 'visualizadas', 'vendidas', 'open_rate_pct', 'conversao_pct', 'conversao_total_pct']


,emitidas,visualizadas,vendidas,open_rate_pct,conversao_pct,conversao_total_pct
0,70907,35772.0,4601.0,50.4,12.9,6.5


In [26]:
# ── Q5.2 · Distribuição de vendas por canal ───────────────────────────────────
sql("""
SELECT
    canalvenda,
    COUNT(DISTINCT idprescricao)  AS prescricoes,
    ROUND(COUNT(DISTINCT idprescricao) * 100.0
          / SUM(COUNT(DISTINCT idprescricao)) OVER (), 1) AS pct_total
FROM prescricao
GROUP BY 1
ORDER BY 2 DESC
""")

3 linhas  |  ['canalvenda', 'prescricoes', 'pct_total']


,canalvenda,prescricoes,pct_total
0,não convertido,66306,93.5
1,farmacia fisica,3888,5.5
2,marketplace,728,1.0


In [27]:
# ── Q5.3 · Conversão por canal (somente visualizadas) ────────────────────────
# Farmácia física vs marketplace vs não convertido
sql("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(visualizadapaciente::INT)  AS visualizada,
        MAX(CASE WHEN itemvendido > 0 THEN 1 ELSE 0 END) AS vendida,
        -- canal: pega o canal de venda que não é 'não convertido'
        COALESCE(
            MAX(CASE WHEN canalvenda != 'não convertido' THEN canalvenda END),
            'não convertido'
        ) AS canal_final
    FROM prescricao
    GROUP BY 1
)
SELECT
    canal_final,
    COUNT(*)          AS prescricoes,
    SUM(visualizada)  AS visualizadas,
    SUM(vendida)      AS vendidas,
    ROUND(SUM(vendida) * 100.0 / NULLIF(SUM(visualizada), 0), 1) AS conversao_pct
FROM presc_nivel
GROUP BY 1
ORDER BY 3 DESC
""")

3 linhas  |  ['canal_final', 'prescricoes', 'visualizadas', 'vendidas', 'conversao_pct']


,canal_final,prescricoes,visualizadas,vendidas,conversao_pct
0,não convertido,66306,32141.0,0.0,0.0
1,farmacia fisica,3873,2903.0,3873.0,133.4
2,marketplace,728,728.0,728.0,100.0


In [28]:
# ── Q5.4 · Evolução semanal das vendas por canal ─────────────────────────────
sql("""
SELECT
    DATE_TRUNC('week', dataprescricao::TIMESTAMP) AS semana,
    canalvenda,
    COUNT(DISTINCT idprescricao) AS prescricoes
FROM prescricao
WHERE canalvenda != 'não convertido'
GROUP BY 1, 2
ORDER BY 1, 2
""")

32 linhas  |  ['semana', 'canalvenda', 'prescricoes']


,semana,canalvenda,prescricoes
0,2024-12-30,farmacia fisica,117
1,2024-12-30,marketplace,18
2,2025-01-06,farmacia fisica,263
3,2025-01-06,marketplace,41
4,2025-01-13,farmacia fisica,246
5,2025-01-13,marketplace,44
6,2025-01-20,farmacia fisica,245
7,2025-01-20,marketplace,40
8,2025-01-27,farmacia fisica,244
9,2025-01-27,marketplace,54


---
## Q6 — Insights Adicionais

In [29]:
# ── Q6.1 · Top 20 medicamentos mais prescritos ───────────────────────────────
sql("""
SELECT
    med.nome,
    med.antimicrobiano,
    med.controleespecial,
    med.mip,
    COUNT(DISTINCT p.idprescricao)  AS prescricoes,
    ROUND(COUNT(DISTINCT p.idprescricao) * 100.0
          / SUM(COUNT(DISTINCT p.idprescricao)) OVER (), 1) AS pct
FROM prescricao p
LEFT JOIN medicamentos med ON p.idmedicamento = med.idmedicamento
GROUP BY 1, 2, 3, 4
ORDER BY 5 DESC
LIMIT 20
""")

20 linhas  |  ['nome', 'antimicrobiano', 'controleespecial', 'mip', 'prescricoes', 'pct']


,nome,antimicrobiano,controleespecial,mip,prescricoes,pct
0,Paracetamol,False,True,True,12860,17.5
1,Glifage XR,False,False,False,8920,12.1
2,Aerolin Spray,False,False,False,6484,8.8
3,Avamys,False,False,False,6250,8.5
4,Sulfato de Salbutamol,False,False,False,5855,8.0
5,Budesonida (Spray),False,False,False,5676,7.7
6,Acetilcisteína,False,False,True,4810,6.5
7,Diprospan,False,False,False,4631,6.3
8,Fluconazol,False,False,False,3576,4.9
9,Cloridrato de Fluoxetina,False,True,False,1707,2.3


In [30]:
# ── Q6.2 · Proporção de medicamentos controlados / antimicrobianos / MIP ─────
sql("""
SELECT
    med.antimicrobiano,
    med.controleespecial,
    med.mip,
    COUNT(DISTINCT p.idprescricao) AS prescricoes
FROM prescricao p
LEFT JOIN medicamentos med ON p.idmedicamento = med.idmedicamento
GROUP BY 1, 2, 3
ORDER BY 4 DESC
""")

5 linhas  |  ['antimicrobiano', 'controleespecial', 'mip', 'prescricoes']


,antimicrobiano,controleespecial,mip,prescricoes
0,False,False,False,44003
1,False,True,True,12860
2,False,True,False,6439
3,False,False,True,4810
4,True,True,False,4409


In [31]:
# ── Q6.3 · Taxa de conversão por medicamento ─────────────────────────────────
sql("""
WITH presc_med AS (
    SELECT
        p.idprescricao,
        med.nome,
        MAX(p.visualizadapaciente::INT)                  AS visualizada,
        MAX(CASE WHEN p.itemvendido > 0 THEN 1 ELSE 0 END) AS vendida
    FROM prescricao p
    LEFT JOIN medicamentos med ON p.idmedicamento = med.idmedicamento
    GROUP BY 1, 2
)
SELECT
    nome,
    COUNT(*)          AS prescricoes,
    SUM(visualizada)  AS visualizadas,
    SUM(vendida)      AS vendidas,
    ROUND(AVG(visualizada) * 100, 1) AS open_rate_pct,
    ROUND(SUM(vendida) * 100.0 / NULLIF(SUM(visualizada), 0), 1) AS conversao_pct
FROM presc_med
GROUP BY 1
HAVING COUNT(*) >= 50
ORDER BY 2 DESC
LIMIT 15
""")

15 linhas  |  ['nome', 'prescricoes', 'visualizadas', 'vendidas', 'open_rate_pct', 'conversao_pct']


,nome,prescricoes,visualizadas,vendidas,open_rate_pct,conversao_pct
0,Paracetamol,12860,7062.0,369.0,54.9,5.2
1,Glifage XR,8920,4000.0,345.0,44.8,8.6
2,Aerolin Spray,6484,3238.0,304.0,49.9,9.4
3,Avamys,6250,3375.0,298.0,54.0,8.8
4,Sulfato de Salbutamol,5855,2470.0,255.0,42.2,10.3
5,Budesonida (Spray),5676,3228.0,268.0,56.9,8.3
6,Acetilcisteína,4810,2468.0,336.0,51.3,13.6
7,Diprospan,4631,2303.0,239.0,49.7,10.4
8,Fluconazol,3576,1939.0,196.0,54.2,10.1
9,Cloridrato de Fluoxetina,1707,989.0,323.0,57.9,32.7


In [32]:
# ── Q6.4 · Tempo entre emissão e venda (janela de conversão) ─────────────────
sql("""
WITH vendas AS (
    SELECT
        idprescricao,
        MIN(dataprescricao::TIMESTAMP) AS emitida_em,
        MIN(datavenda)                 AS vendida_em
    FROM prescricao
    WHERE datavenda IS NOT NULL
    GROUP BY 1
),
tempo AS (
    SELECT
        idprescricao,
        DATEDIFF('hour', emitida_em, vendida_em) AS horas_ate_venda
    FROM vendas
    WHERE vendida_em >= emitida_em  -- remove negativos
)
SELECT
    COUNT(*)                                        AS vendas_totais,
    ROUND(MEDIAN(horas_ate_venda), 0)               AS mediana_horas,
    ROUND(AVG(horas_ate_venda), 1)                  AS media_horas,
    SUM(CASE WHEN horas_ate_venda <= 6  THEN 1 ELSE 0 END) AS ate_6h,
    SUM(CASE WHEN horas_ate_venda <= 24 THEN 1 ELSE 0 END) AS ate_24h,
    SUM(CASE WHEN horas_ate_venda <= 72 THEN 1 ELSE 0 END) AS ate_72h,
    ROUND(SUM(CASE WHEN horas_ate_venda <= 6  THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_ate_6h,
    ROUND(SUM(CASE WHEN horas_ate_venda <= 24 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_ate_24h
FROM tempo
""")

1 linhas  |  ['vendas_totais', 'mediana_horas', 'media_horas', 'ate_6h', 'ate_24h', 'ate_72h', 'pct_ate_6h', 'pct_ate_24h']


,vendas_totais,mediana_horas,media_horas,ate_6h,ate_24h,ate_72h,pct_ate_6h,pct_ate_24h
0,4586,8.0,77.0,2193.0,2980.0,3600.0,47.8,65.0


In [33]:
# ── Q6.5 · Distribuição por faixa de tempo até a venda ───────────────────────
sql("""
WITH vendas AS (
    SELECT
        idprescricao,
        DATEDIFF('hour',
            MIN(dataprescricao::TIMESTAMP),
            MIN(datavenda::TIMESTAMP)
        ) AS horas
    FROM prescricao
    WHERE datavenda IS NOT NULL
    GROUP BY 1
    HAVING MIN(datavenda::TIMESTAMP) >= MIN(dataprescricao::TIMESTAMP)
)
SELECT
    CASE
        WHEN horas < 1   THEN '< 1h'
        WHEN horas < 6   THEN '1–6h'
        WHEN horas < 24  THEN '6–24h'
        WHEN horas < 72  THEN '1–3 dias'
        WHEN horas < 168 THEN '3–7 dias'
        ELSE '> 7 dias'
    END AS faixa_tempo,
    COUNT(*) AS vendas,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct
FROM vendas
GROUP BY 1
ORDER BY MIN(horas)
""")

6 linhas  |  ['faixa_tempo', 'vendas', 'pct']


,faixa_tempo,vendas,pct
0,< 1h,580,12.6
1,1–6h,1513,33.0
2,6–24h,838,18.3
3,1–3 dias,652,14.2
4,3–7 dias,412,9.0
5,> 7 dias,591,12.9


In [34]:
# ── Q6.6 · Retenção de médicos: quantos prescreveram N vezes ─────────────────
sql("""
WITH medico_cnt AS (
    SELECT
        idmedico,
        COUNT(DISTINCT idprescricao) AS num_prescricoes
    FROM prescricao
    GROUP BY 1
)
SELECT
    CASE
        WHEN num_prescricoes = 1  THEN '1 (churn)'
        WHEN num_prescricoes <= 3 THEN '2–3'
        WHEN num_prescricoes <= 10 THEN '4–10'
        WHEN num_prescricoes <= 30 THEN '11–30'
        ELSE '> 30 (top)'
    END AS faixa,
    COUNT(*)  AS medicos,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_medicos,
    SUM(num_prescricoes) AS prescricoes_geradas
FROM medico_cnt
GROUP BY 1
ORDER BY MIN(num_prescricoes)
""")

5 linhas  |  ['faixa', 'medicos', 'pct_medicos', 'prescricoes_geradas']


,faixa,medicos,pct_medicos,prescricoes_geradas
0,1 (churn),6620,41.8,6620.0
1,2–3,4329,27.3,10219.0
2,4–10,3410,21.5,20258.0
3,11–30,1223,7.7,20285.0
4,> 30 (top),249,1.6,13525.0


In [36]:
# ── Q6.7 · Prescrições por convênio ──────────────────────────────────────────
sql("""
SELECT
    CASE
        WHEN idconvenio IS NULL THEN 'Sem Convênio'
        ELSE 'Convênio ' || CAST(CAST(idconvenio AS INT) AS VARCHAR)
    END AS tipo_atendimento,
    COUNT(DISTINCT idprescricao) AS prescricoes,
    ROUND(COUNT(DISTINCT idprescricao) * 100.0
          / SUM(COUNT(DISTINCT idprescricao)) OVER (), 1) AS pct
FROM prescricao
GROUP BY 1
ORDER BY 2 DESC
LIMIT 10
""")

10 linhas  |  ['tipo_atendimento', 'prescricoes', 'pct']


,tipo_atendimento,prescricoes,pct
0,Sem Convênio,41949,59.2
1,Convênio 981873,4508,6.4
2,Convênio 296156,3341,4.7
3,Convênio 1725637,468,0.7
4,Convênio 1656285,365,0.5
5,Convênio 54,350,0.5
6,Convênio 1113267,219,0.3
7,Convênio 474369,169,0.2
8,Convênio 1216942,114,0.2
9,Convênio 1710832,83,0.1


---
## Q7 — Recomendações Operacionais (SQLs de Suporte)

In [38]:
# ── Q7.1 · Receitas não abertas por estado — prioridade de notificação ────────
sql("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(upper(estadopaciente))            AS estado,
        MAX(visualizadapaciente::INT)  AS visualizada
    FROM prescricao
    GROUP BY 1
)
SELECT
    estado,
    COUNT(*)                                         AS total,
    SUM(1 - visualizada)                             AS nao_abertas,
    ROUND(SUM(1 - visualizada) * 100.0 / COUNT(*), 1) AS taxa_nao_abertura_pct
FROM presc_nivel
GROUP BY 1
HAVING COUNT(*) >= 30
ORDER BY taxa_nao_abertura_pct DESC
LIMIT 15
""")

15 linhas  |  ['estado', 'total', 'nao_abertas', 'taxa_nao_abertura_pct']


,estado,total,nao_abertas,taxa_nao_abertura_pct
0,TO,233,161.0,69.1
1,AM,538,364.0,67.7
2,SE,435,282.0,64.8
3,MG,3806,2446.0,64.3
4,ES,4054,2535.0,62.5
5,PR,3995,2429.0,60.8
6,RN,1359,825.0,60.7
7,RO,97,52.0,53.6
8,BA,968,507.0,52.4
9,SP,34720,17766.0,51.2


In [39]:
# ── Q7.2 · Identificar prescrições em janela de remarketing (6–24h, não abertas)
# Este SQL seria usado em produção para alimentar um job de notificação
sql("""
WITH ultima_prescricao AS (
    SELECT
        idprescricao,
        idpaciente,
        MAX(dataprescricao::TIMESTAMP)  AS emitida_em,
        MAX(visualizadapaciente::INT)   AS visualizada
    FROM prescricao
    GROUP BY 1, 2
)
SELECT
    COUNT(*) AS prescricoes_elegíveis_para_remarketing
FROM ultima_prescricao
WHERE visualizada = 0
  AND DATEDIFF('hour', emitida_em, CURRENT_TIMESTAMP) BETWEEN 6 AND 24
""")

1 linhas  |  ['prescricoes_elegíveis_para_remarketing']


,prescricoes_elegíveis_para_remarketing
0,0


In [40]:
# ── Q7.3 · Médicos em risco de churn (1 prescrição, > 30 dias sem nova) ───────
sql("""
WITH medico_atividade AS (
    SELECT
        idmedico,
        COUNT(DISTINCT idprescricao)               AS total_prescricoes,
        MAX(dataprescricao::TIMESTAMP)             AS ultima_prescricao,
        DATEDIFF('day', MAX(dataprescricao::TIMESTAMP), CURRENT_TIMESTAMP) AS dias_inativo
    FROM prescricao
    GROUP BY 1
)
SELECT
    COUNT(*) AS medicos_risco_churn
FROM medico_atividade
WHERE total_prescricoes = 1
  AND dias_inativo > 30
""")

1 linhas  |  ['medicos_risco_churn']


,medicos_risco_churn
0,6620


In [41]:
# ── Q7.4 · Oportunidade marketplace: visualizadas não convertidas ─────────────
sql("""
WITH presc_nivel AS (
    SELECT
        idprescricao,
        MAX(visualizadapaciente::INT) AS visualizada,
        MAX(CASE WHEN itemvendido > 0 THEN 1 ELSE 0 END) AS vendida
    FROM prescricao
    GROUP BY 1
)
SELECT
    COUNT(*)                                         AS visualizadas_nao_convertidas,
    -- Potencial de receita: se converter 10% delas via marketplace
    ROUND(COUNT(*) * 0.10)                           AS ganho_potencial_10pct
FROM presc_nivel
WHERE visualizada = 1
  AND vendida = 0
""")

1 linhas  |  ['visualizadas_nao_convertidas', 'ganho_potencial_10pct']


,visualizadas_nao_convertidas,ganho_potencial_10pct
0,32141,3214.0


In [42]:
# ── Q7.5 · Top especialidades sem cadastro de especialidade (qualificação) ────
sql("""
SELECT
    m.conselhoprofissional,
    m.ufconselho,
    COUNT(DISTINCT p.idprescricao) AS prescricoes,
    COUNT(DISTINCT m.idmedico)     AS medicos_sem_especialidade
FROM prescricao p
LEFT JOIN medicos m ON p.idmedico = m.idmedico
WHERE UPPER(m.especialidade) = 'SEM ESPECIALIDADE'
GROUP BY 1, 2
ORDER BY 3 DESC
LIMIT 10
""")

10 linhas  |  ['conselhoprofissional', 'ufconselho', 'prescricoes', 'medicos_sem_especialidade']


,conselhoprofissional,ufconselho,prescricoes,medicos_sem_especialidade
0,CRM,SP,6737,1235
1,CRM,MG,850,126
2,CRM,GO,712,68
3,CRM - RJ,RJ,681,234
4,CRM,PR,667,112
5,CRM,RJ,604,90
6,CRM,DF,409,47
7,CRM,SC,361,68
8,CRM,RS,361,62
9,CRM,BA,230,41


---
## Bonus — KPIs Consolidados (Dashboard Summary Card)

In [43]:
# ── KPI Summary Table — todos os indicadores em uma query ────────────────────
sql("""
WITH base AS (
    SELECT
        COUNT(DISTINCT idprescricao)   AS total_prescricoes,
        COUNT(DISTINCT idpaciente)     AS total_pacientes,
        COUNT(DISTINCT idmedico)       AS total_medicos,
        COUNT(DISTINCT CAST(dataprescricao AS DATE)) AS dias_com_movimento
    FROM prescricao
),
presc_nivel AS (
    SELECT
        idprescricao,
        MAX(visualizadapaciente::INT) AS visualizada,
        MAX(CASE WHEN itemvendido > 0 THEN 1 ELSE 0 END) AS vendida
    FROM prescricao
    GROUP BY 1
),
funnel AS (
    SELECT
        COUNT(*)          AS emitidas,
        SUM(visualizada)  AS visualizadas,
        SUM(vendida)      AS vendidas
    FROM presc_nivel
)
SELECT
    b.total_prescricoes,
    b.total_pacientes,
    b.total_medicos,
    ROUND(b.total_prescricoes * 1.0 / b.dias_com_movimento, 0) AS media_diaria,
    f.visualizadas,
    ROUND(f.visualizadas * 100.0 / f.emitidas, 1) AS open_rate_pct,
    f.vendidas,
    ROUND(f.vendidas * 100.0 / NULLIF(f.visualizadas, 0), 1) AS conversao_pct,
    ROUND(f.vendidas * 100.0 / f.emitidas, 1) AS conversao_total_pct
FROM base b, funnel f
""")

1 linhas  |  ['total_prescricoes', 'total_pacientes', 'total_medicos', 'media_diaria', 'visualizadas', 'open_rate_pct', 'vendidas', 'conversao_pct', 'conversao_total_pct']


,total_prescricoes,total_pacientes,total_medicos,media_diaria,visualizadas,open_rate_pct,vendidas,conversao_pct,conversao_total_pct
0,70907,68767,15831,657.0,35772.0,50.4,4601.0,12.9,6.5


---
## Notas de Adaptação para Outros Engines

| Função DuckDB | Snowflake | BigQuery | Databricks SQL |
|---|---|---|---|
| `DATE_TRUNC('week', ts)` | `DATE_TRUNC('week', ts)` | `DATE_TRUNC(ts, WEEK)` | `DATE_TRUNC('week', ts)` |
| `DAYNAME(ts)` | `DAYNAME(ts)` | `FORMAT_DATE('%A', DATE(ts))` | `DATE_FORMAT(ts, 'EEEE')` |
| `DAYOFWEEK(ts)` | `DAYOFWEEK(ts)` | `DAYOFWEEK(DATE(ts))` | `DAYOFWEEK(ts)` |
| `HOUR(ts)` | `HOUR(ts)` | `EXTRACT(HOUR FROM ts)` | `HOUR(ts)` |
| `DATEDIFF('hour', a, b)` | `DATEDIFF('hour', a, b)` | `TIMESTAMP_DIFF(b, a, HOUR)` | `TIMESTAMPDIFF(HOUR, a, b)` |
| `MEDIAN(col)` | `MEDIAN(col)` | `APPROX_QUANTILES(col,2)[OFFSET(1)]` | `PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY col)` |
| `read_csv_auto(...)` | — (use COPY/STAGE) | — (use LOAD DATA) | — (use spark.read) |

Para rodar no **Databricks**, substitua as views por:
```python
spark.read.csv('dbfs:/path/prescricaomedicamento.csv', header=True).createOrReplaceTempView('prescricao')
```
e troque `con.execute(sql).df()` por `spark.sql(sql).toPandas()`.
